# Model Surgery

Surgically modify model weights to transplant components between models,
permanently knock out heads, and compare corresponding components across
models. All operations are functional — they return new models without
mutating the originals.

This notebook covers the `irtk.model_surgery` module.

## Why This Matters

Model surgery tools let you transplant components between models, zero out specific heads or layers, and compare head behavior across architectures. This enables controlled experiments that isolate the contribution of individual components.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import model_surgery

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Knockout Head Weights

Permanently zero out a head's output weights. This is more efficient
than runtime ablation when evaluating across many prompts.

In [ ]:
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)

paris_id = tokenizer.encode(" Paris")[0]

# Clean prediction
clean_logits = model(tokens)
clean_prob = float(jax.nn.softmax(clean_logits[-1])[paris_id])
print(f"Clean P(' Paris'): {clean_prob:.4f}")

# Knockout individual heads and check effect
print("\nHead knockout effects (L9):")
for head in range(model.cfg.n_heads):
    ko_model = model_surgery.knockout_head_weights(model, layer=9, head=head)
    logits = ko_model(tokens)
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    delta = prob - clean_prob
    bar = '█' * int(abs(delta) / 0.01) if abs(delta) > 0.001 else ''
    print(f"  L9H{head:2d}: P=' Paris'={prob:.4f} (Δ={delta:+.4f}) {bar}")

## 2. Zero Out Layer

Make an entire layer act as a skip connection (identity through the
residual stream). Useful for measuring layer importance.

In [ ]:
# Zero out each layer and measure effect
layer_effects = []
for layer in range(model.cfg.n_layers):
    zeroed = model_surgery.zero_out_layer(model, layer=layer, component="both")
    logits = zeroed(tokens)
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    layer_effects.append(prob - clean_prob)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['red' if x < 0 else 'green' for x in layer_effects]
ax.bar(range(model.cfg.n_layers), layer_effects, color=colors)
ax.set_xlabel("Layer")
ax.set_ylabel("Δ P(' Paris')")
ax.set_title("Effect of Zeroing Each Layer")
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# Separate attn vs MLP
attn_effects = []
mlp_effects = []
for layer in range(model.cfg.n_layers):
    zeroed_attn = model_surgery.zero_out_layer(model, layer=layer, component="attn")
    logits = zeroed_attn(tokens)
    attn_effects.append(float(jax.nn.softmax(logits[-1])[paris_id]) - clean_prob)

    zeroed_mlp = model_surgery.zero_out_layer(model, layer=layer, component="mlp")
    logits = zeroed_mlp(tokens)
    mlp_effects.append(float(jax.nn.softmax(logits[-1])[paris_id]) - clean_prob)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(model.cfg.n_layers)
ax.bar(x - 0.2, attn_effects, 0.4, label="Attn zeroed", color='steelblue')
ax.bar(x + 0.2, mlp_effects, 0.4, label="MLP zeroed", color='coral')
ax.set_xlabel("Layer")
ax.set_ylabel("Δ P(' Paris')")
ax.set_title("Attention vs MLP Layer Importance")
ax.axhline(0, color='black', linewidth=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Transplant Heads

Copy attention head weights from one model to another. Here we'll
create a variant model with some heads zeroed, then transplant
the original heads back one by one.

In [ ]:
# Create a "broken" model with layer 9 attention zeroed
broken = model_surgery.zero_out_layer(model, layer=9, component="attn")
broken_logits = broken(tokens)
broken_prob = float(jax.nn.softmax(broken_logits[-1])[paris_id])
print(f"Broken model P(' Paris'): {broken_prob:.4f}")
print(f"Clean model P(' Paris'):  {clean_prob:.4f}")

# Transplant heads back one at a time
print("\nTransplanting L9 heads back:")
for head in range(model.cfg.n_heads):
    restored = model_surgery.transplant_heads(model, broken, [(9, head)])
    logits = restored(tokens)
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    recovery = prob - broken_prob
    print(f"  +L9H{head:2d}: P=' Paris'={prob:.4f} (recovery={recovery:+.4f})")

## 4. Transplant MLP

Replace an MLP layer from one model variant into another.

In [ ]:
# Zero out MLP at layer 8, then transplant it back
broken_mlp = model_surgery.zero_out_layer(model, layer=8, component="mlp")
broken_mlp_logits = broken_mlp(tokens)
broken_mlp_prob = float(jax.nn.softmax(broken_mlp_logits[-1])[paris_id])

restored = model_surgery.transplant_mlp(model, broken_mlp, layer=8)
restored_logits = restored(tokens)
restored_prob = float(jax.nn.softmax(restored_logits[-1])[paris_id])

print(f"Clean:             P=' Paris'={clean_prob:.4f}")
print(f"MLP L8 zeroed:     P=' Paris'={broken_mlp_prob:.4f}")
print(f"MLP L8 restored:   P=' Paris'={restored_prob:.4f}")

# Verify: transplanting from clean into broken should fully restore
assert abs(restored_prob - clean_prob) < 1e-4, "Transplant should fully restore!"
print("\nTransplant fully restores the MLP!")

## 5. Compare Heads Across Models

Measure how similar corresponding heads are between two model variants.
Useful for understanding what changed during fine-tuning.

In [ ]:
# Create a "perturbed" model by adding noise to one layer
import equinox as eqx

noise = jax.random.normal(jax.random.PRNGKey(0), model.blocks[5].attn.W_Q.shape) * 0.01
noisy_W_Q = model.blocks[5].attn.W_Q + noise
perturbed = eqx.tree_at(lambda m: m.blocks[5].attn.W_Q, model, noisy_W_Q)

# Compare each head in the perturbed layer
print("Head comparison L5 (original vs perturbed):")
for head in range(model.cfg.n_heads):
    result = model_surgery.compare_heads_across_models(model, perturbed, layer=5, head=head)
    print(f"  H{head}: overall_cos={result['overall_cosine']:.6f}  "
          f"W_Q_cos={result['W_Q_cosine']:.6f}  W_Q_l2={result['W_Q_l2']:.6f}")

# Compare unperturbed layer (should be identical)
print("\nHead comparison L4 (should be identical):")
for head in range(min(4, model.cfg.n_heads)):
    result = model_surgery.compare_heads_across_models(model, perturbed, layer=4, head=head)
    print(f"  H{head}: overall_cos={result['overall_cosine']:.6f}")

In [ ]:
# Visualize head similarity across all layers
similarities = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        result = model_surgery.compare_heads_across_models(model, perturbed, layer, head)
        similarities[layer, head] = result["overall_cosine"]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(similarities, cmap="RdYlGn", aspect="auto", vmin=0.99, vmax=1.0)
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Head Cosine Similarity (original vs perturbed at L5)")
plt.colorbar(im, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `transplant_heads()` | Copy attention head weights between models |
| `transplant_mlp()` | Copy MLP weights between models |
| `knockout_head_weights()` | Zero a head's output weights permanently |
| `compare_heads_across_models()` | Cosine similarity between corresponding heads |
| `zero_out_layer()` | Make a layer produce zero output (skip only) |